# 04 – Export Features for Machine Learning

Extract a feature matrix from EEG data that you can feed directly into scikit-learn or any ML framework.

**Pipeline:**
1. Load session (CSV or WebSocket)
2. Slide a fixed-length window across the data
3. For each window × channel, extract: band powers, variance, zero-crossings, peak-to-peak
4. Export to `features.csv`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

SAMPLE_RATE = 250
WIN_SEC = 2       # window length
STEP_SEC = 1      # step (overlap = WIN - STEP)

BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta":  (12, 30),
    "gamma": (30, 100),
}

## Load data

In [ ]:
import glob

csv_files = sorted(glob.glob("../recordings/pieeg_*.csv"))
df = pd.read_csv(csv_files[-1])
df["time"] = df["timestamp"] - df["timestamp"].iloc[0]
ch_cols = [c for c in df.columns if c.startswith("ch")]
print(f"{len(ch_cols)} channels, {len(df)} samples, {df['time'].iloc[-1]:.1f} s")

## (Alternative) Stream from PiEEG server

In [ ]:
# import asyncio, json, websockets
#
# async def collect(seconds=60):
#     rows = []
#     async with websockets.connect("ws://localhost:1616") as ws:
#         welcome = json.loads(await ws.recv())
#         target = seconds * SAMPLE_RATE
#         while len(rows) < target:
#             msg = json.loads(await ws.recv())
#             if "channels" in msg:
#                 rows.append([msg["t"]] + msg["channels"])
#     cols = ["timestamp"] + [f"ch{i+1}" for i in range(welcome["channels"])]
#     return pd.DataFrame(rows, columns=cols)
#
# df = await collect(60)
# df["time"] = df["timestamp"] - df["timestamp"].iloc[0]
# ch_cols = [c for c in df.columns if c.startswith("ch")]

## Feature extraction functions

In [ ]:
def band_power(data, fs, band):
    """Absolute band power via Welch."""
    nperseg = min(len(data), fs * 2)
    freqs, psd = signal.welch(data, fs=fs, nperseg=nperseg)
    lo, hi = band
    mask = (freqs >= lo) & (freqs <= hi)
    return np.trapz(psd[mask], freqs[mask])


def zero_crossings(data):
    """Count zero crossings."""
    return np.sum(np.diff(np.sign(data)) != 0)


def extract_features(segment, fs):
    """Extract features from a single-channel segment."""
    feats = {}
    # Band powers
    for name, rng in BANDS.items():
        feats[f"bp_{name}"] = band_power(segment, fs, rng)
    # Time-domain
    feats["variance"] = np.var(segment)
    feats["std"] = np.std(segment)
    feats["ptp"] = np.ptp(segment)          # peak-to-peak
    feats["zero_cross"] = zero_crossings(segment)
    feats["rms"] = np.sqrt(np.mean(segment ** 2))
    # Ratio features
    total = sum(feats[f"bp_{b}"] for b in BANDS) + 1e-12
    feats["alpha_ratio"] = feats["bp_alpha"] / total
    feats["theta_beta_ratio"] = feats["bp_theta"] / (feats["bp_beta"] + 1e-12)
    return feats

## Slide windows & build feature matrix

In [ ]:
win_samples = int(WIN_SEC * SAMPLE_RATE)
step_samples = int(STEP_SEC * SAMPLE_RATE)

records = []
start = 0
while start + win_samples <= len(df):
    row = {"time": df["time"].values[start]}
    for ch in ch_cols:
        seg = df[ch].values[start : start + win_samples]
        feats = extract_features(seg, SAMPLE_RATE)
        for k, v in feats.items():
            row[f"{ch}_{k}"] = v
    records.append(row)
    start += step_samples

feat_df = pd.DataFrame(records)
print(f"Feature matrix: {feat_df.shape[0]} windows × {feat_df.shape[1]} columns")
feat_df.head()

## Quick sanity check – feature distributions

In [ ]:
alpha_cols = [c for c in feat_df.columns if c.endswith("_bp_alpha")]

fig, axes = plt.subplots(1, min(4, len(alpha_cols)), figsize=(14, 3))
for ax, col in zip(axes, alpha_cols[:4]):
    ax.hist(feat_df[col], bins=20, color="#2ecc71", edgecolor="white")
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("Power")
plt.suptitle("Alpha band power distributions", fontsize=12)
plt.tight_layout()
plt.show()

## Correlation heatmap (ch1 features)

In [ ]:
ch1_feat_cols = [c for c in feat_df.columns if c.startswith("ch1_")]
corr = feat_df[ch1_feat_cols].corr()

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(ch1_feat_cols)))
ax.set_xticklabels([c.replace("ch1_", "") for c in ch1_feat_cols],
                    rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(ch1_feat_cols)))
ax.set_yticklabels([c.replace("ch1_", "") for c in ch1_feat_cols], fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Feature Correlations – ch1")
plt.tight_layout()
plt.show()

## Export to CSV

In [ ]:
out_path = "features.csv"
feat_df.to_csv(out_path, index=False)
print(f"Saved {feat_df.shape[0]} rows × {feat_df.shape[1]} cols → {out_path}")

## Quick ML demo – train / test split with Random Forest

For a real experiment you'd add a **label** column (e.g. "eyes_open" vs "eyes_closed").
Here we just show the sklearn pipeline shape.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# -------------------------------------------------------------------
# In a real experiment: load features.csv that includes a label column.
#   feat_df = pd.read_csv("features.csv")
#   X = feat_df.drop(columns=["time", "label"]).values
#   y = feat_df["label"].values
# -------------------------------------------------------------------

# Demo: create a synthetic binary label (above / below median alpha on ch1)
X = feat_df.drop(columns=["time"]).values
y = (feat_df["ch1_bp_alpha"] > feat_df["ch1_bp_alpha"].median()).astype(int).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

score = clf.score(X_test, y_test)
print(f"Test accuracy: {score:.2%}")
print(f"\nTop 5 features:")
feat_names = feat_df.drop(columns=["time"]).columns
for idx in np.argsort(clf.feature_importances_)[-5:][::-1]:
    print(f"  {feat_names[idx]:35s}  {clf.feature_importances_[idx]:.4f}")